In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

pd.set_option('display.max_columns', None)

os.chdir("..")
print(os.getcwd())

competition = "Premier League"
season = "2015-2016"

project_dir = Path.cwd()
processed_dir = project_dir / "data/processed" / competition / season
features_path = processed_dir / "player_match_features.parquet"

model_dir = project_dir / "data/model" / season
perc_pos_path = model_dir / "position_percentile_features.parquet"

# proc_features = pd.read_parquet(features_path)
perc_pos_features = pd.read_parquet(perc_pos_path)

/Users


FileNotFoundError: [Errno 2] No such file or directory: '/Users/data/model/2015-2016/position_percentile_features.parquet'

In [4]:
proc_features.head(5)

,match_id,player_id,player,team_id,team,started,time_on,time_off,minutes_played,passes_attempted,passes_completed,progressive_passes,shot_assists,assists,xA,dribbles_attempted,dribbles_completed,carries,progressive_carries,shots,shots_on_target,xG,goals,duels,tackles_attempted,tackles_won,poss_won_final_third_tackles,clearances,blocks,interceptions,interceptions_won,poss_won_final_third_interceptions,ball_recoveries_attempted,ball_recoveries_won,pressures,fouls
0,3754189,3522,Heurelho da Silva Gomes,23,Watford,True,0.0,5652.305,94.205083,39.0,26.0,16.0,0.0,0.0,0.000000,0.0,0.0,15.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,8.0,0.0,0.0
1,3754189,3991,Allan Romeo Nyom,23,Watford,True,0.0,5652.305,94.205083,62.0,45.0,5.0,1.0,0.0,0.005593,1.0,1.0,47.0,5.0,1.0,0.0,0.018391,0.0,6.0,5.0,2.0,0.0,2.0,1.0,0.0,0.0,0.0,5.0,5.0,9.0,0.0
2,3754189,3309,Sebastian Prödl,23,Watford,True,0.0,5652.305,94.205083,20.0,14.0,2.0,0.0,0.0,0.000000,0.0,0.0,10.0,0.0,0.0,0.0,0.000000,0.0,1.0,0.0,0.0,0.0,5.0,0.0,1.0,0.0,0.0,2.0,2.0,1.0,1.0
3,3754189,3263,Craig Cathcart,23,Watford,True,0.0,5652.305,94.205083,32.0,22.0,4.0,0.0,0.0,0.000000,0.0,0.0,21.0,0.0,0.0,0.0,0.000000,0.0,3.0,1.0,1.0,0.0,5.0,1.0,1.0,0.0,0.0,1.0,1.0,5.0,0.0
4,3754189,3725,José Holebas,23,Watford,True,0.0,2829.238,47.153967,38.0,23.0,3.0,0.0,0.0,0.000000,2.0,0.0,27.0,2.0,2.0,0.0,0.030909,0.0,2.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,5.0,5.0,4.0,2.0


In [5]:
season_totals = (
    proc_features.groupby("player_id", as_index=False)
    .agg(
        # Descriptive fields
        player=("player", "first"),
        team_ids=("team_id", lambda x: list(x.dropna().unique())),
        teams=("team", lambda x: list(x.dropna().unique())),

        # Playing time
        matches_played=("match_id", "nunique"),
        starts=("started", "sum"),
        minutes_played=("minutes_played", "sum"),

        # Passing / creation
        passes_attempted=("passes_attempted", "sum"),
        passes_completed=("passes_completed", "sum"),
        progressive_passes=("progressive_passes", "sum"),
        shot_assists=("shot_assists", "sum"),
        assists=("assists", "sum"),
        xA=("xA", "sum"),

        # Dribbling / carrying
        dribbles_attempted=("dribbles_attempted", "sum"),
        dribbles_completed=("dribbles_completed", "sum"),
        carries=("carries", "sum"),
        progressive_carries=("progressive_carries", "sum"),

        # Shooting
        shots=("shots", "sum"),
        shots_on_target=("shots_on_target", "sum"),
        xG=("xG", "sum"),
        goals=("goals", "sum"),

        # Defending
        duels=("duels", "sum"),
        tackles_attempted=("tackles_attempted", "sum"),
        tackles_won=("tackles_won", "sum"),
        poss_won_final_third_tackles=(
            "poss_won_final_third_tackles", "sum"
        ),
        clearances=("clearances", "sum"),
        blocks=("blocks", "sum"),
        interceptions=("interceptions", "sum"),
        interceptions_won=("interceptions_won", "sum"),
        poss_won_final_third_interceptions=(
            "poss_won_final_third_interceptions", "sum"
        ),
        ball_recoveries_attempted=(
            "ball_recoveries_attempted", "sum"
        ),
        ball_recoveries_won=(
            "ball_recoveries_won", "sum"
        ),

        # Pressing / discipline
        pressures=("pressures", "sum"),
        fouls=("fouls", "sum"),
    )
)

In [6]:
def sanity_check_season_totals(df):
    print("=== SEASON TOTALS SANITY CHECK ===\n")

    # --------------------------------------------------
    # 1. BASIC SHAPE / GRAIN
    # --------------------------------------------------

    print("--- BASIC INFO ---")
    print(f"Rows / players: {len(df):,}")
    print(f"Unique player_ids: {df['player_id'].nunique():,}")
    print(f"Duplicate player_ids: {df['player_id'].duplicated().sum():,}")
    print()

    assert df["player_id"].notna().all(), "ERROR: Null player_ids"
    assert df["player_id"].is_unique, "ERROR: player_id is not unique"

    # --------------------------------------------------
    # 2. NULLS
    # --------------------------------------------------

    print("--- NULL COUNTS ---")

    nulls = df.isna().sum()
    nulls = nulls[nulls > 0].sort_values(ascending=False)

    if len(nulls):
        print(nulls)
    else:
        print("No null values found.")
    print()

    # --------------------------------------------------
    # 3. NEGATIVE VALUES
    # --------------------------------------------------

    numeric_cols = df.select_dtypes(include=np.number).columns
    stat_cols = [c for c in numeric_cols if c not in ["player_id"]]
    negative_counts = (df[stat_cols] < 0).sum()
    negative_counts = negative_counts[negative_counts > 0]

    print("--- NEGATIVE VALUES ---")

    if len(negative_counts):
        print(negative_counts)
    else:
        print("No negative values found.")
    print()

    # --------------------------------------------------
    # 4. LOGICAL RELATIONSHIPS
    # --------------------------------------------------

    checks = {
        "passes_completed > passes_attempted":
            df["passes_completed"] > df["passes_attempted"],
        "dribbles_completed > dribbles_attempted":
            df["dribbles_completed"] > df["dribbles_attempted"],
        "shots_on_target > shots":
            df["shots_on_target"] > df["shots"],
        "goals > shots":
            df["goals"] > df["shots"],
        "tackles_won > tackles_attempted":
            df["tackles_won"] > df["tackles_attempted"],
        "interceptions_won > interceptions":
            df["interceptions_won"] > df["interceptions"],
        "ball_recoveries_won > ball_recoveries_attempted":
            df["ball_recoveries_won"] > df["ball_recoveries_attempted"],
        "starts > matches_played":
            df["starts"] > df["matches_played"],
    }

    print("--- LOGICAL CHECKS ---")

    for name, mask in checks.items():
        count = mask.sum()
        if count:
            print(f"WARNING: {name}: {count} players")
        else:
            print(f"PASS: {name}")
    print()

    # --------------------------------------------------
    # 5. PLAYING TIME
    # --------------------------------------------------

    print("--- PLAYING TIME ---")
    print(
        df["minutes_played"]
        .describe()
        .round(1)
    )
    print("\nPlayers with > 3,500 minutes:")
    print(
        df.loc[
            df["minutes_played"] > 3500,
            ["player", "teams", "minutes_played"]
        ].sort_values("minutes_played", ascending=False)
    )

    # --------------------------------------------------
    # 6. LEADERBOARD SPOT CHECKS
    # --------------------------------------------------

    print("\n--- TOP GOALSCORERS ---")
    print(
        df[
            ["player", "teams", "goals", "xG"]
        ]
        .sort_values("goals", ascending=False)
        .head(10)
        .to_string(index=False)
    )

    print("\n--- TOP ASSISTS ---")
    print(
        df[
            ["player", "teams", "assists", "xA"]
        ]
        .sort_values("assists", ascending=False)
        .head(10)
        .to_string(index=False)
    )

    print("\n--- TOP xG ---")
    print(
        df[
            ["player", "teams", "xG", "goals"]
        ]
        .sort_values("xG", ascending=False)
        .head(10)
        .to_string(index=False)
    )

    print("\n--- TOP xA ---")
    print(
        df[
            ["player", "teams", "xA", "assists"]
        ]
        .sort_values("xA", ascending=False)
        .head(10)
        .to_string(index=False)
    )

    print("\n=== CHECK COMPLETE ===")

In [3]:
sanity_check_season_totals(season_totals)

NameError: name 'sanity_check_season_totals' is not defined

In [2]:
import os
os.chdir("..")
print(os.getcwd())
from src.features import build_player_season_features

build_player_season_features(competition_id=2, season_id=27)

/Users/TeddyKolios/personal proj
Compiling season-long features for competition: Premier League, season: 2015-2016....
Saved 550 player-season rows to /Users/TeddyKolios/personal proj/statsbomb/data/processed/Premier League/2015-2016/player_season_features.parquet
